In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
GTSDB_ROOT = PROJECT_ROOT / "FullIJCNN2013"
RUNS_DIR = PROJECT_ROOT / "runs"
MODELS_DIR = PROJECT_ROOT / "models"

print("Project root:", PROJECT_ROOT)
print("GTSDB exists:", GTSDB_ROOT.exists())

In [ ]:
## Nach Neustart ausführen

import torch

DATA_YAML = DATA_DIR / "yolo" / "gtsdb" / "data.yaml"

assert DATA_YAML.exists(), f"data.yaml nicht gefunden: {DATA_YAML}"

print("Project root:", PROJECT_ROOT)
print("Using data.yaml:", DATA_YAML)
print("MPS available:", torch.backends.mps.is_available())

In [ ]:
from ultralytics import YOLO

COMMON = dict(
    data=str(DATA_YAML),
    epochs=120,
    imgsz=960,
    batch=16,         
    device="mps",
    seed=42,
    patience=30,
    pretrained=True,
    augment=True,
    close_mosaic=10,
    workers=4,
    project=str(PROJECT_ROOT / "runs" / "traffic_signs"),
)

In [ ]:
model_s = YOLO("yolov8s.pt")
model_s.train(
    name="v8s_run3_opt3",
    **COMMON
)

In [ ]:
from ultralytics import YOLO

BEST_S = RUNS_DIR / "detect" / "runs" / "traffic_signs" / "v8s_run3_opt3" / "weights" / "best.pt"

model_s_best = YOLO(str(BEST_S))

metrics_s = model_s_best.val(
    data=str(DATA_YAML),
    imgsz=960,
    device="mps",
    conf=0.25,
    iou=0.6
)

metrics_s

In [ ]:
import yaml
from pathlib import Path

d = yaml.safe_load(open(DATA_YAML, "r"))

yaml_root = Path(d["path"])
root = yaml_root if yaml_root.is_absolute() else (PROJECT_ROOT / yaml_root)
val_dir = root / d["val"]  

pred = model_s_best.predict(
    source=str(val_dir),
    imgsz=960,
    conf=0.25,
    device="mps",
    save=True
)

pred

In [ ]:
import time
import random
import torch
from ultralytics import YOLO

VAL_DIR = DATA_DIR / "yolo" / "gtsdb" / "images" / "val"

baseline_w = RUNS_DIR / "gtsdb_yolov8n_baseline" / "weights" / "best.pt"
exp2_w     = RUNS_DIR / "gtsdb_yolov8n_img800" / "weights" / "best.pt"

imgs = list(VAL_DIR.glob("*.png"))
random.seed(42)
sample = random.sample(imgs, k=min(50, len(imgs)))

def benchmark(weights_path, imgsz: int):
    model = YOLO(str(weights_path))
    device = "mps" if torch.backends.mps.is_available() else "cpu"

    # warmup
    _ = model.predict(source=str(sample[0]), imgsz=imgsz, device=device, verbose=False)

    t0 = time.time()
    for p in sample:
        _ = model.predict(source=str(p), imgsz=imgsz, device=device, verbose=False)
    t1 = time.time()

    total = t1 - t0
    ms_per_img = (total / len(sample)) * 1000
    fps = 1000 / ms_per_img
    return ms_per_img, fps

b_ms, b_fps = benchmark(baseline_w, 640)
e_ms, e_fps = benchmark(exp2_w, 800)

print("Baseline imgsz640:", f"{b_ms:.2f} ms/img", f"~{b_fps:.1f} FPS")
print("Exp2 imgsz800:", f"{e_ms:.2f} ms/img", f"~{e_fps:.1f} FPS")

## Videos die getestet werden

In [ ]:
from ultralytics import YOLO

MODEL = RUNS_DIR / "detect" / "runs" / "traffic_signs" / "v8s_run3_opt3" / "weights" / "best.pt"
VIDEO = DATA_DIR / "demo" / "dashcam_55s.mp4"

model = YOLO(str(MODEL))

pred = model.predict(
    source=str(VIDEO),
    imgsz=960,
    conf=0.25,
    device="mps",
    save=True
)

pred

In [ ]:
from ultralytics import YOLO

MODEL = RUNS_DIR / "detect" / "runs" / "traffic_signs" / "v8s_run3_opt3" / "weights" / "best.pt"
VIDEO = DATA_DIR / "demo" / "berlin_clip_9m_20s.mp4"

model = YOLO(str(MODEL))

pred = model.predict(
    source=str(VIDEO),
    imgsz=960,
    conf=0.25,
    device="mps",
    save=True
)

pred

In [ ]:
from ultralytics import YOLO

MODEL = RUNS_DIR / "detect" / "runs" / "traffic_signs" / "v8s_run3_opt3" / "weights" / "best.pt"
VIDEO = DATA_DIR / "demo" / "Jesingen.mp4"

model = YOLO(str(MODEL))

pred = model.predict(
    source=str(VIDEO),
    imgsz=960,
    conf=0.25,
    device="mps",
    save=True
)

pred

In [ ]:
from ultralytics import YOLO

MODEL = RUNS_DIR / "detect" / "runs" / "traffic_signs" / "v8s_run3_opt3" / "weights" / "best.pt"
VIDEO = DATA_DIR / "demo" / "Jesingennacht.mp4"

model = YOLO(str(MODEL))

pred = model.predict(
    source=str(VIDEO),
    imgsz=960,
    conf=0.25,
    device="mps",
    save=True
)

pred

In [ ]:
from ultralytics import YOLO

MODEL = RUNS_DIR / "detect" / "runs" / "traffic_signs" / "v8s_run3_opt3" / "weights" / "best.pt"
VIDEO = DATA_DIR / "demo" / "NT_Neuffen_VID_20220617.mp4"

model = YOLO(str(MODEL))

pred = model.predict(
    source=str(VIDEO),
    imgsz=960,
    conf=0.25,
    device="mps",
    save=True
)

pred